# Auth & Session Management

> Seamless Solveit authentication using your real Chrome (or Vivaldi) browser so Google OAuth works natively, with persistent `.auth/` session state reusable across all `fastcdp` scripts, tests, and notebooks in this repository.

In [ ]:
#| default_exp auth

In [ ]:
#| export
import asyncio, time
from urllib.parse import urlparse
from fastcore.net import urljson
from fastcore.utils import *

In [ ]:
#| export
def _find_repo_root():
    in_file = '__file__' in globals()
    start = (Path(__file__).parent if in_file else Path.cwd()).resolve()
    is_root = lambda p: (p/'pyproject.toml').is_file() or (p/'CRAFT.js').is_file()
    return next((p for p in (start, *start.parents) if is_root(p)), start.parent if in_file else start)

REPO_ROOT = _find_repo_root()
DEFAULT_AUTH_FILE = REPO_ROOT/'.auth'/'solveit-state.json'
DEFAULT_CONFIG_FILE = REPO_ROOT/'.auth'/'config.json'
DEFAULT_CHROME_PROFILE = REPO_ROOT/'.auth'/'chrome-solveit-profile'
DEFAULT_VIVALDI_PROFILE = REPO_ROOT/'.auth'/'vivaldi-profile'
DEFAULT_DEBUG_DIR = REPO_ROOT/'.craft-debug'
DEFAULT_SOLVEIT_URL = 'https://solve.it.com/dashboard'

VIVALDI_MAC_PATH = Path('/Applications/Vivaldi.app/Contents/MacOS/Vivaldi')
CHROME_MAC_PATH = Path('/Applications/Google Chrome.app/Contents/MacOS/Google Chrome')

## Config & Browser Detection

Session cookies and the remembered default Solveit URL live under `.auth/` (gitignored, permissions `0600`).

In [ ]:
#| export
def load_saved_config():
    try: return DEFAULT_CONFIG_FILE.read_json()
    except Exception: return {}

def save_config(updates):
    cfg = merge(load_saved_config(), filter_values(updates, lambda v: v is not None))
    DEFAULT_CONFIG_FILE.write_json(cfg, indent=2)

def default_target_url(): return os.environ.get('SOLVEIT_TEST_URL') or load_saved_config().get('default_url')

In [ ]:
#| export
def _check_executable(executable_path):
    path = Path(executable_path)
    if not path.is_file(): raise SystemExit(f"Browser executable not found: {path}")
    return path

def detect_native_browser(browser='chrome', executable_path=None):
    "Find a real desktop Chromium-based browser (Chrome by default, or Vivaldi) for seamless Google OAuth."
    if executable_path: return _check_executable(executable_path)
    cands = [VIVALDI_MAC_PATH, CHROME_MAC_PATH] if browser == 'vivaldi' else [CHROME_MAC_PATH, VIVALDI_MAC_PATH]
    if p := first(p for p in cands if p.is_file()): return p
    raise SystemExit("Could not find Google Chrome or Vivaldi. Pass --executable-path to specify your browser.")

def resolve_browser_executable(browser='chrome', executable_path=None):
    "Resolve optional browser executable path for `fastcdp` (`chrome`, `vivaldi`, or `chromium` default)."
    if executable_path: return str(_check_executable(executable_path))
    if browser == 'chromium': return None
    target = VIVALDI_MAC_PATH if browser == 'vivaldi' else CHROME_MAC_PATH
    if not target.is_file():
        raise SystemExit(f"{browser.title()} not found at {target}. Pass --executable-path to specify its location.")
    return str(target)

## Authentication Checks & Cookie Parsers

In [ ]:
#| export
_LOGIN_PATHS = ('/login', '/signin', '/auth', '/oauth')
_LOGIN_PHRASES = ('no access. please login', 'please login and then retry')
_GOOGLE_AUTH_PHRASES = ('sign in with google', "couldn't sign you in")

def looks_like_login_url(url): return any(t in urlparse(url).path.lower() for t in _LOGIN_PATHS)

async def is_authenticated_page(page, status=None):
    "Check URL, HTTP status, and Solveit's 403 login notice on a `fastcdp` page."
    url = await page.eval('location.href') or ''
    if looks_like_login_url(url): return False
    if status in (401, 403): return False
    try:
        body = (await page.sel_text('body') or '')[:200].lower()
        if any(p in body for p in _LOGIN_PHRASES): return False
    except Exception: pass
    return True

In [ ]:
import tempfile
from fastcore.test import test_eq, test_fail

`looks_like_login_url` flags common OAuth and sign-in paths while ignoring normal Solveit dialog URLs.

In [ ]:
test_eq(looks_like_login_url('https://example.solve.it.com/oauth/callback'), True)
test_eq(looks_like_login_url('https://example.solve.it.com/dialog/42'), False)

In [ ]:
#| export
_SAME_SITE_MAP = {'strict': 'Strict', 'lax': 'Lax', 'none': 'None', 'no_restriction': 'None', 'unspecified': 'Lax'}

def _write_storage_state(cookies, auth_file):
    auth_file = Path(auth_file)
    auth_file.parent.mkdir(parents=True, exist_ok=True)
    auth_file.write_json({'cookies': cookies, 'origins': []}, indent=2)
    os.chmod(auth_file, 0o600)
    return cookies

def parse_cookie_header(header, url):
    "Parse a raw HTTP `Cookie` header (single-line, multi-line DevTools copy, or cURL) into cookie dicts."
    parsed = urlparse(url)
    host = parsed.hostname
    if not host: raise ValueError(f"Could not extract hostname from URL: {url!r}")
    domain = '.solve.it.com' if host == 'solve.it.com' or host.endswith('.solve.it.com') else host
    secure = parsed.scheme == 'https'

    text = header.strip()
    if text.lower().startswith('curl '):
        m = re.search(r"""(?:-b|--cookie)\s+['"]([^'"]+)['"]""", text) or re.search(
            r"""-H\s+['"]cookie:\s*([^'"]+)['"]""", text, re.IGNORECASE)
        if m: text = m.group(1).strip()

    lines = L(text.splitlines()).map(Self.strip()).filter()
    if lines and lines[0].lower() == 'cookie:': lines = lines[1:]
    raw = '; '.join(lines)
    if raw.lower().startswith('cookie:'): raw = raw.split(':', 1)[1].strip()

    cookies = []
    for part in raw.split(';'):
        part = part.strip()
        if not part or '=' not in part: continue
        name, value = part.split('=', 1)
        if not (name := name.strip()): continue
        cookies.append({'name': name, 'value': value.strip(), 'domain': domain, 'path': '/',
                        'expires': -1, 'httpOnly': False, 'secure': secure, 'sameSite': 'Lax'})
    return cookies

def save_cookie_header(header, url, auth_file=DEFAULT_AUTH_FILE):
    cookies = parse_cookie_header(header, url)
    if not cookies: raise ValueError("Could not parse any name=value cookies from input.")
    return _write_storage_state(cookies, auth_file)

def import_cookies_json(cookies_path, auth_file=DEFAULT_AUTH_FILE):
    raw = Path(cookies_path).read_json()
    if isinstance(raw, dict) and 'cookies' in raw: cookies = raw['cookies']
    elif isinstance(raw, list): cookies = raw
    else: raise ValueError("Expected a JSON list of cookies or a storage_state object.")
    normalized = [{
        'name': c['name'], 'value': c['value'], 'domain': c['domain'], 'path': c.get('path', '/'),
        'expires': float(c.get('expires', c.get('expirationDate', -1))),
        'httpOnly': bool(c.get('httpOnly', False)), 'secure': bool(c.get('secure', True)),
        'sameSite': _SAME_SITE_MAP.get(str(c.get('sameSite', 'Lax')).lower(), 'Lax'),
    } for c in cookies]
    return _write_storage_state(normalized, auth_file)

def cookies_for_cdp(auth_file):
    "Load saved cookies from `auth_file` in CDP `Storage.setCookies` format."
    raw = Path(auth_file).read_json()
    items = raw['cookies'] if isinstance(raw, dict) else raw
    out = []
    for c in items:
        d = {k: c[k] for k in ('name', 'value', 'domain', 'path', 'httpOnly', 'secure', 'sameSite') if c.get(k) is not None}
        if (exp := c.get('expires', -1)) and float(exp) > 0: d['expires'] = float(exp)
        out.append(d)
    return out

`parse_cookie_header` handles DevTools multiline copies and `curl` snippets, `import_cookies_json` normalizes browser-extension exports into `.auth/` state files with `0600` permissions, and `cookies_for_cdp` adapts them for CDP `Storage.setCookies`.

In [ ]:
url = 'https://example.solve.it.com'
test_eq([c['name'] for c in parse_cookie_header("cookie:\nsid=xyz; foo=bar", url)], ['sid', 'foo'])
test_eq([c['name'] for c in parse_cookie_header("curl 'https://example.solve.it.com' -b 'a=1; b=2'", url)], ['a', 'b'])
test_fail(lambda: parse_cookie_header('sid=1', 'not-a-url'), ValueError)

with tempfile.TemporaryDirectory() as d:
    src, dst = Path(d)/'in.json', Path(d)/'out.json'
    src.write_json([{'name': 'sid', 'value': '1', 'domain': 'example.solve.it.com',
                     'expirationDate': 1700000000, 'sameSite': 'no_restriction'}])
    out = import_cookies_json(src, dst)
    test_eq((out[0]['expires'], out[0]['sameSite']), (1700000000.0, 'None'))
    test_eq(dst.stat().st_mode & 0o777, 0o600)
    cdp_c = cookies_for_cdp(dst)
    test_eq((cdp_c[0]['name'], cdp_c[0]['expires']), ('sid', 1700000000.0))

## Seamless One-Step Chrome/Vivaldi Login via `fastcdp`

In [ ]:
#| export
def _is_instance_url(url):
    host = (urlparse(url).hostname or '').lower()
    return host.endswith('.solve.it.com') and host != 'solve.it.com'

async def _page_is_solveit_authenticated(page, target_host):
    try:
        url = await page.eval('location.href') or ''
        host = (urlparse(url).hostname or '').lower()
        if not host.endswith('solve.it.com'): return False
        if target_host and target_host != 'solve.it.com' and host != target_host: return False
        if looks_like_login_url(url): return False
        body = (await page.sel_text('body') or '')[:300].lower()
        return bool(body.strip()) and not any(p in body for p in (*_LOGIN_PHRASES, *_GOOGLE_AUTH_PHRASES))
    except Exception: return False

def _ws_from_cdp_url(cdp_url):
    if cdp_url.startswith('ws'): return cdp_url
    return urljson(f"{cdp_url.rstrip('/')}/json/version")['webSocketDebuggerUrl']

async def alogin(
    url=None, # Optional Solveit URL (defaults to `https://solve.it.com/dashboard` for account-wide `.solve.it.com` auth)
    *,
    auth_file=DEFAULT_AUTH_FILE, # Path to write saved cookie state JSON
    profile_dir=DEFAULT_CHROME_PROFILE, # Dedicated Solveit Chrome user-data directory
    browser='chrome', # Preferred native browser (`chrome` or `vivaldi`)
    executable_path=None, # Explicit path to browser binary
    cdp_url=None, # Existing Chrome DevTools Protocol endpoint to attach to directly
    timeout=300, # Seconds to wait for Solveit authentication to complete
):
    "Authenticate with Solveit via `fastcdp` on the dedicated Chrome profile (or existing CDP endpoint) and save cookies."
    from fastcdp import CDP

    auth_file = Path(auth_file)
    target_url = url or DEFAULT_SOLVEIT_URL
    target_host = (urlparse(target_url).hostname or '').lower() or None

    if cdp_url:
        cdp = await CDP.connect(wsconn=_ws_from_cdp_url(cdp_url))
        try: cookies = (await cdp.Storage.getCookies())['cookies']
        finally: await cdp.close()
        _write_storage_state(cookies, auth_file)
        if _is_instance_url(target_url): save_config({'default_url': target_url})
        return auth_file

    chrome = detect_native_browser(browser if browser != 'chromium' else 'chrome', executable_path)
    cdp = await CDP.launch(user_data_dir=profile_dir, headless=False, chrome=chrome)
    try:
        page = await cdp.new_page()
        await page.goto(target_url, wait=None)
        deadline, authed_url = time.monotonic() + timeout, None
        while time.monotonic() < deadline:
            if cdp.proc is not None and cdp.proc.returncode is not None:
                raise SystemExit("Browser was closed before Solveit login completed.")
            for t in await cdp.pages:
                if 'solve.it.com' not in t.get('url', ''): continue
                try:
                    p = await cdp.attach_page(t['targetId'])
                    if await _page_is_solveit_authenticated(p, target_host):
                        await asyncio.sleep(1.0)
                        cookies = (await cdp.Storage.getCookies())['cookies']
                        _write_storage_state(cookies, auth_file)
                        authed_url = t['url']
                        break
                except Exception: pass
            if authed_url: break
            await asyncio.sleep(0.5)
    finally:
        if getattr(cdp, 'proc', None) is not None: await cdp.quit()
        else: await cdp.close()

    if not authed_url: raise SystemExit(f"Timed out after {timeout}s waiting for Solveit login.")
    if _is_instance_url(target_url): save_config({'default_url': target_url})
    return auth_file

def login(*args, **kwargs):
    "Synchronous wrapper around `alogin`."
    return asyncio.run(alogin(*args, **kwargs))

def ensure_auth(auth_file=DEFAULT_AUTH_FILE, *, auto_login=False, url=None):
    "Return `auth_file` if present; optionally trigger Chrome login when missing."
    auth_file = Path(auth_file)
    if auth_file.is_file(): return auth_file
    if auto_login: return login(url=url, auth_file=auth_file)
    raise SystemExit(
        f"Auth state not found at {auth_file}.\n"
        f"Run: .venv/bin/craft-cdp auth --url {url or DEFAULT_SOLVEIT_URL}")